# Task 3 AI Resume Screening

# Name : Mithilesh Kolhapurkar


In [3]:
%pip install -q -U langchain langchain-core langchain-groq langsmith pydantic pandas gradio plotly pymupdf pillow requests


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.8/108.8 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 472.0/472.0 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 52.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 3.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.2 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.4.4 which is incompatible.


In [4]:
import os, re, json, uuid, base64, requests
import fitz
import pandas as pd
import gradio as gr
import plotly.express as px
from getpass import getpass
from typing import List, Any
from pydantic import BaseModel, Field
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_groq import ChatGroq


if not os.getenv("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass("Enter GROQ_API_KEY: ")

if not os.getenv("LANGCHAIN_API_KEY"):
    v = input("Enable LangSmith tracing? (y/n): ").strip().lower()
    if v == "y":
        os.environ["LANGCHAIN_API_KEY"] = getpass("Enter LANGCHAIN_API_KEY: ")

os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
os.environ.setdefault("LANGSMITH_TRACING", "true")
os.environ.setdefault("LANGCHAIN_PROJECT", "resume-screening-studio-pro")


Enter GROQ_API_KEY: ··········
Enable LangSmith tracing? (y/n): y
Enter LANGCHAIN_API_KEY: ··········


'resume-screening-studio-pro'

In [5]:
class CandidateProfile(BaseModel):
    candidate_name: str = "Unknown"
    skills: List[str] = Field(default_factory=list)
    tools: List[str] = Field(default_factory=list)
    years_experience: float = 0
    education: List[str] = Field(default_factory=list)
    projects: List[str] = Field(default_factory=list)
    domain_experience: List[str] = Field(default_factory=list)
    certifications: List[str] = Field(default_factory=list)

class JobRequirements(BaseModel):
    role: str = "Unknown"
    must_have_skills: List[str] = Field(default_factory=list)
    good_to_have_skills: List[str] = Field(default_factory=list)
    tools: List[str] = Field(default_factory=list)
    minimum_years_experience: float = 0
    education_requirements: List[str] = Field(default_factory=list)
    domain: List[str] = Field(default_factory=list)

class MatchResult(BaseModel):
    matched_must_have_skills: List[str] = Field(default_factory=list)
    missing_must_have_skills: List[str] = Field(default_factory=list)
    matched_good_to_have_skills: List[str] = Field(default_factory=list)
    matched_tools: List[str] = Field(default_factory=list)
    missing_tools: List[str] = Field(default_factory=list)
    experience_gap: float = 0
    education_match: bool = False
    domain_match: List[str] = Field(default_factory=list)

class Explanation(BaseModel):
    summary: str
    strengths: List[str]
    gaps: List[str]
    final_recommendation: str

class ScreeningResult(BaseModel):
    candidate_name: str
    role: str
    score: int
    match: MatchResult
    explanation: Explanation


In [19]:
EXTRACTION_PROMPT = PromptTemplate.from_template("""
You are an expert technical recruiter.
Extract only facts explicitly stated in the resume text.
Do not infer or hallucinate missing information.
Return strict JSON with this schema:
{{
  "candidate_name": string,
  "skills": [string],
  "tools": [string],
  "years_experience": number,
  "education": [string],
  "projects": [string],
  "domain_experience": [string],
  "certifications": [string]
}}
Rules:
- Include only skills/tools directly mentioned.
- If missing, use empty list or 0 for years_experience.
- Normalize duplicates.
- Output JSON only.
Resume text:
{resume_text}
""")

JOB_PROMPT = PromptTemplate.from_template("""
You are an expert hiring analyst.
Extract explicit requirements from the job description.
Do not invent requirements.
Return strict JSON with this schema:
{{
  "role": string,
  "must_have_skills": [string],
  "good_to_have_skills": [string],
  "tools": [string],
  "minimum_years_experience": number,
  "education_requirements": [string],
  "domain": [string]
}}
Rules:
- If requirement is not explicit, do not include it.
- Normalize duplicates.
- Output JSON only.
Job description text:
{job_description}
""")

EXPLANATION_PROMPT = PromptTemplate.from_template("""
You are preparing recruiter-facing justification.
Use only provided structured facts.
Do not add any skill or experience not present in evidence.
Return strict JSON with this schema:
{{
  "summary": string,
  "strengths": [string],
  "gaps": [string],
  "final_recommendation": string
}}
Inputs:
Candidate profile:
{candidate_json}
Job requirements:
{job_json}
Matching evidence:
{match_json}
Final score:
{score}
Recommendation must be one of: Strong Shortlist, Consider, Reject.
Output JSON only.
""")

In [7]:
if not os.getenv("HF_TOKEN"):
    use_hf = input("Enable Hugging Face OCR API? (y/n): ").strip().lower()
    if use_hf == "y":
        os.environ["HF_TOKEN"] = getpass("Enter HF_TOKEN: ")

HF_OCR_MODEL = "microsoft/trocr-base-printed"

def _hf_ocr_image_bytes(img_bytes: bytes) -> str:
    token = os.getenv("HF_TOKEN", "")
    headers = {"Authorization": f"Bearer {token}"} if token else {}
    url = f"https://api-inference.huggingface.co/models/{HF_OCR_MODEL}"
    r = requests.post(url, headers=headers, data=img_bytes, timeout=120)
    r.raise_for_status()
    data = r.json()
    if isinstance(data, list) and data and "generated_text" in data[0]:
        return data[0]["generated_text"].strip()
    if isinstance(data, dict) and "generated_text" in data:
        return data["generated_text"].strip()
    return ""

def extract_text_from_pdf(path: str) -> str:
    doc = fitz.open(path)
    chunks = []
    for page in doc:
        t = page.get_text("text").strip()
        if t:
            chunks.append(t)
    text_layer = "\n\n".join(chunks).strip()
    if len(text_layer) >= 300:
        return text_layer

    ocr_chunks = []
    for page in doc:
        pix = page.get_pixmap(dpi=220, alpha=False)
        img_bytes = pix.tobytes("png")
        txt = _hf_ocr_image_bytes(img_bytes)
        if txt:
            ocr_chunks.append(txt)
    return "\n\n".join(ocr_chunks).strip()


Enable Hugging Face OCR API? (y/n): y
Enter HF_TOKEN: ··········


In [15]:
class ResumeScreeningPipeline:
    def __init__(self, model_name="llama-3.3-70b-versatile"):
        self.llm = ChatGroq(model=model_name, temperature=0)

    def run(self, resume_text: str, job_description: str, tag: str = ""):
        # 1. Extract Profile
        profile_json = self.llm.invoke(EXTRACTION_PROMPT.format(resume_text=resume_text)).content
        profile = CandidateProfile.model_validate_json(self._clean_json(profile_json))

        # 2. Extract Job Requirements
        job_json = self.llm.invoke(JOB_PROMPT.format(job_description=job_description)).content
        job = JobRequirements.model_validate_json(self._clean_json(job_json))

        # 3. Calculate Match (Logic)
        matched_must = [s for s in profile.skills if any(m.lower() in s.lower() for m in job.must_have_skills)]
        missing_must = [m for m in job.must_have_skills if not any(m.lower() in s.lower() for s in profile.skills)]

        score = 100
        if job.must_have_skills:
            score = int((len(matched_must) / len(job.must_have_skills)) * 100)

        match_res = MatchResult(
            matched_must_have_skills=matched_must,
            missing_must_have_skills=missing_must,
            experience_gap=max(0, job.minimum_years_experience - profile.years_experience)
        )

        # 4. Generate Explanation
        expl_json = self.llm.invoke(EXPLANATION_PROMPT.format(
            candidate_json=profile.model_dump_json(),
            job_json=job.model_dump_json(),
            match_json=match_res.model_dump_json(),
            score=score
        )).content
        explanation = Explanation.model_validate_json(self._clean_json(expl_json))

        return ScreeningResult(
            candidate_name=profile.candidate_name,
            role=job.role,
            score=score,
            match=match_res,
            explanation=explanation
        )

    def _clean_json(self, text):
        match = re.search(r'\{.*\}', text, re.DOTALL)
        return match.group(0) if match else text

In [23]:
from google.colab import files
import os

# --- Step 1: Input Job Description ---
job_description = """
Enter your job description here
"""

# --- Step 2: Upload Resume ---
print("Please upload your resume PDF:")
uploaded = files.upload()

if uploaded:
    filename = list(uploaded.keys())[0]
    resume_pdf_path = os.path.join(os.getcwd(), filename)

    print(f"\nProcessing uploaded resume: {filename}...")
    resume_text = extract_text_from_pdf(resume_pdf_path)

    # --- Step 3: Run Analysis ---
    pipeline = ResumeScreeningPipeline(model_name="llama-3.3-70b-versatile")
    result = pipeline.run(resume_text, job_description)

    print("\n--- SCREENING RESULT ---")
    print(f"Candidate: {result.candidate_name}")
    print(f"Score: {result.score}/100")
    print(f"Recommendation: {result.explanation.final_recommendation}")
    print(f"Summary: {result.explanation.summary}")
    print("\nStrengths:")
    for s in result.explanation.strengths: print(f"- {s}")
    print("\nGaps:")
    for g in result.explanation.gaps: print(f"- {g}")
else:
    print("No file was uploaded. Please run the cell again to upload your resume.")

Please upload your resume PDF:


Saving sbcyynmtpnyd.pdf to sbcyynmtpnyd.pdf

Processing uploaded resume: sbcyynmtpnyd.pdf...

--- SCREENING RESULT ---
Candidate: Harshibar
Score: 100/100
Recommendation: Strong Shortlist
Summary: Harshibar has 4 years of experience with a strong set of skills including Python, JavaScript, React.js, and SQL, as well as experience with various tools such as Figma, Notion, and GitHub.

Strengths:
- Proficient in multiple programming languages
- Experienced with a range of tools and technologies
- Completed various projects including Hyku Consulting, Minimal Icon Pack, and CommonIntern

Gaps:
- No domain experience listed
- No certifications listed
- Education match is false


In [24]:
from google.colab import files
import os

# --- Step 1: Input Job Description ---
job_description = """
Enter your job description here
"""

# --- Step 2: Upload Resume ---
print("Please upload your resume PDF:")
uploaded = files.upload()

if uploaded:
    filename = list(uploaded.keys())[0]
    resume_pdf_path = os.path.join(os.getcwd(), filename)

    print(f"\nProcessing uploaded resume: {filename}...")
    resume_text = extract_text_from_pdf(resume_pdf_path)

    # --- Step 3: Run Analysis ---
    pipeline = ResumeScreeningPipeline(model_name="llama-3.3-70b-versatile")
    result = pipeline.run(resume_text, job_description)

    print("\n--- SCREENING RESULT ---")
    print(f"Candidate: {result.candidate_name}")
    print(f"Score: {result.score}/100")
    print(f"Recommendation: {result.explanation.final_recommendation}")
    print(f"Summary: {result.explanation.summary}")
    print("\nStrengths:")
    for s in result.explanation.strengths: print(f"- {s}")
    print("\nGaps:")
    for g in result.explanation.gaps: print(f"- {g}")
else:
    print("No file was uploaded. Please run the cell again to upload your resume.")

Please upload your resume PDF:


Saving trgqjpwnmtgv.pdf to trgqjpwnmtgv.pdf

Processing uploaded resume: trgqjpwnmtgv.pdf...

--- SCREENING RESULT ---
Candidate: YOUR NAME HERE
Score: 100/100
Recommendation: Consider
Summary: Candidate has a strong educational background with a Ph.D., M.Sc., and B.Sc., and possesses skills in C++, Embedded Systems, and Statistical Analysis, but lacks experience and domain-specific knowledge.

Strengths:
- Strong educational background
- Skills in C++, Embedded Systems, and Statistical Analysis

Gaps:
- Lack of experience
- No domain-specific experience


In [25]:
from google.colab import files
import os

# --- Step 1: Input Job Description ---
job_description = """
Enter your job description here
"""

# --- Step 2: Upload Resume ---
print("Please upload your resume PDF:")
uploaded = files.upload()

if uploaded:
    filename = list(uploaded.keys())[0]
    resume_pdf_path = os.path.join(os.getcwd(), filename)

    print(f"\nProcessing uploaded resume: {filename}...")
    resume_text = extract_text_from_pdf(resume_pdf_path)

    # --- Step 3: Run Analysis ---
    pipeline = ResumeScreeningPipeline(model_name="llama-3.3-70b-versatile")
    result = pipeline.run(resume_text, job_description)

    print("\n--- SCREENING RESULT ---")
    print(f"Candidate: {result.candidate_name}")
    print(f"Score: {result.score}/100")
    print(f"Recommendation: {result.explanation.final_recommendation}")
    print(f"Summary: {result.explanation.summary}")
    print("\nStrengths:")
    for s in result.explanation.strengths: print(f"- {s}")
    print("\nGaps:")
    for g in result.explanation.gaps: print(f"- {g}")
else:
    print("No file was uploaded. Please run the cell again to upload your resume.")

Please upload your resume PDF:


Saving jsmpwkcwyntg.pdf to jsmpwkcwyntg.pdf

Processing uploaded resume: jsmpwkcwyntg.pdf...

--- SCREENING RESULT ---
Candidate: HARSH GADGIL
Score: 100/100
Recommendation: Strong Shortlist
Summary: Candidate HARSH GADGIL has 8 years of experience with a strong technical background in skills such as C, C++, Java, Python, and Machine Learning, as well as experience with tools like Kafka, Spark, and Docker.

Strengths:
- Diverse technical skill set
- Experience with big data tools like Kafka and Spark
- Proficient in multiple programming languages

Gaps:
- No domain experience specified
- No certifications listed
